# Hypergraph GNN + MILP: Weighted Set Cover

Set Cover doğal bir hypergraph problemidir: **eleman = node**, **aday küme = hyperedge**. Amaç tüm elemanları kapsayan en düşük maliyetli setleri seçmektir.

Bu örnek: full MILP → optimal set etiketleri → `HypergraphConv` → set skoru → candidate pruning → coverage repair → reduced MILP akışını uygular. GNN solver'ın yerini almaz; karar uzayını daraltır.

In [ ]:
import random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from scipy.optimize import milp, LinearConstraint, Bounds
from torch_geometric.nn import HypergraphConv

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def generate_instance(n_elements=14,n_sets=22,density=.28,seed=0):
    rng=np.random.default_rng(seed)
    A=rng.random((n_elements,n_sets))<density
    for i in range(n_elements):
        idx=np.flatnonzero(A[i])
        if len(idx)<2:
            A[i,rng.choice(n_sets,size=2-len(idx),replace=False)]=True
    for j in range(n_sets):
        if not A[:,j].any(): A[rng.integers(n_elements),j]=True
    cov=A.sum(0)
    c=rng.uniform(1,5,n_sets)+.65*cov+rng.uniform(0,2,n_sets)
    return {"A":A.astype(float),"c":c.astype(float)}

def solve(inst,allowed=None):
    A,c=inst["A"],inst["c"]; n,m=A.shape
    allowed=np.arange(m) if allowed is None else np.array(sorted(set(allowed)),dtype=int)
    if len(allowed)==0 or np.any(A[:,allowed].sum(1)==0): return None
    con=LinearConstraint(A[:,allowed],np.ones(n),np.full(n,np.inf))
    t=time.perf_counter()
    r=milp(c=c[allowed],integrality=np.ones(len(allowed)),
           bounds=Bounds(np.zeros(len(allowed)),np.ones(len(allowed))),
           constraints=con,options={"time_limit":20.0})
    if not r.success or r.x is None: return None
    x=np.zeros(m); x[allowed]=np.rint(r.x)
    return {"x":x,"obj":float(c@x),"time":time.perf_counter()-t}

ex=generate_instance(seed=SEED); solve(ex)


## Hypergraph temsili

`hyperedge_index[0]` eleman kimliğini, `hyperedge_index[1]` set/hyperedge kimliğini taşır. Böylece bir setin aynı anda birden fazla elemanı kapsadığı higher-order ilişki korunur.

Node features: degree, kapsayan setlerin minimum ve ortalama normalize maliyeti. Set features: normalize maliyet, coverage ratio, cost/coverage.

In [ ]:
def build_graph(inst,label=None):
    A,c=inst["A"],inst["c"]; n,m=A.shape
    rows,cols=np.nonzero(A)
    hi=torch.tensor(np.vstack([rows,cols]),dtype=torch.long)
    cn=c/max(c.max(),1e-9); deg=A.sum(1)/m
    mn=[]; av=[]
    for i in range(n):
        idx=np.flatnonzero(A[i]); mn.append(cn[idx].min()); av.append(cn[idx].mean())
    node_x=np.c_[deg,mn,av].astype("float32")
    cov=A.sum(0)
    set_x=np.c_[cn,cov/n,cn/np.maximum(cov,1)].astype("float32")
    g={"node_x":torch.tensor(node_x),"set_x":torch.tensor(set_x),"hyperedge_index":hi}
    if label is not None: g["y"]=torch.tensor(label,dtype=torch.float32)
    return g

def edge_mean(h,hi,m):
    ni,ei=hi; out=h.new_zeros((m,h.size(1))); out.index_add_(0,ei,h[ni])
    cnt=h.new_zeros((m,1)); cnt.index_add_(0,ei,h.new_ones((len(ei),1)))
    return out/cnt.clamp_min(1)

class HyperSetCover(nn.Module):
    def __init__(self,h=48):
        super().__init__()
        self.lin=nn.Linear(3,h); self.c1=HypergraphConv(h,h); self.c2=HypergraphConv(h,h)
        self.head=nn.Sequential(nn.Linear(h+3,h),nn.ReLU(),nn.Linear(h,1))
    def forward(self,g):
        x=F.relu(self.lin(g["node_x"]))
        x=F.relu(self.c1(x,g["hyperedge_index"]))
        x=F.relu(self.c2(x,g["hyperedge_index"]))
        e=edge_mean(x,g["hyperedge_index"],g["set_x"].size(0))
        return self.head(torch.cat([e,g["set_x"]],1)).squeeze(1)

class SetMLP(nn.Module):
    def __init__(self,h=48):
        super().__init__(); self.net=nn.Sequential(nn.Linear(3,h),nn.ReLU(),nn.Linear(h,1))
    def forward(self,g): return self.net(g["set_x"]).squeeze(1)


In [ ]:
def make_data(N,start):
    out=[]
    for k in range(N):
        inst=generate_instance(seed=start+k); sol=solve(inst)
        if sol: out.append((inst,sol,build_graph(inst,sol["x"])))
    return out
train=make_data(90,1000); test=make_data(30,5000)

def dev(g): return {k:v.to(device) for k,v in g.items()}
def train_model(model,epochs=70):
    model=model.to(device); opt=torch.optim.Adam(model.parameters(),lr=2e-3)
    pos=sum(float(g["y"].sum()) for _,_,g in train); total=sum(g["y"].numel() for _,_,g in train)
    pw=torch.tensor(max((total-pos)/max(pos,1),1),device=device)
    for ep in range(epochs):
        for i in np.random.permutation(len(train)):
            g=dev(train[i][2]); loss=F.binary_cross_entropy_with_logits(model(g),g["y"],pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1)%20==0: print("epoch",ep+1)
    return model
hyper=train_model(HyperSetCover()); mlp=train_model(SetMLP())


## Pruning ve OR değerlendirmesi

Yalnız accuracy ölçmek yerine **optimal-set recall, retained-set ratio, feasibility, objective gap ve solve time** ölçülür. Pruning sonrası her elemanın en az bir set tarafından kapsanmasını sağlayan coverage repair uygulanır.

In [ ]:
@torch.no_grad()
def scores(model,g):
    model.eval(); return torch.sigmoid(model(dev(g))).cpu().numpy()

def repair(inst,chosen,s):
    A=inst["A"]; chosen=set(map(int,chosen))
    for i in range(A.shape[0]):
        cand=np.flatnonzero(A[i])
        if not any(j in chosen for j in cand): chosen.add(int(cand[np.argmax(s[cand])]))
    return np.array(sorted(chosen))

def select(inst,s,frac=.45):
    k=max(1,int(np.ceil(len(s)*frac)))
    return repair(inst,np.argsort(s)[-k:],s)

def heuristic(inst):
    return inst["A"].sum(0)/np.maximum(inst["c"],1e-9)

def evaluate(model=None,baseline=False):
    rows=[]
    for inst,full,g in test:
        s=heuristic(inst) if baseline else scores(model,g)
        keep=select(inst,s); red=solve(inst,keep)
        opt=set(np.flatnonzero(full["x"]>.5)); kept=set(keep.tolist())
        rows.append({
            "feasible":red is not None,
            "recall":len(opt&kept)/max(len(opt),1),
            "retained":len(keep)/len(inst["c"]),
            "gap_pct":np.nan if red is None else 100*(red["obj"]-full["obj"])/max(abs(full["obj"]),1e-9),
            "full_time":full["time"],"reduced_time":np.nan if red is None else red["time"]
        })
    return rows

def summary(name,rows):
    a=lambda k:np.array([r[k] for r in rows],float)
    return {"method":name,"feasibility":a("feasible").mean(),"optimal_set_recall":a("recall").mean(),
            "retained_ratio":a("retained").mean(),"mean_gap_pct":np.nanmean(a("gap_pct")),
            "full_time":a("full_time").mean(),"reduced_time":np.nanmean(a("reduced_time"))}

[summary("Hypergraph GNN",evaluate(hyper)),
 summary("Set-feature MLP",evaluate(mlp)),
 summary("Cost/Coverage heuristic",evaluate(baseline=True))]


## Neden bu gerçekten hypergraph?

Set Cover incidence matrisi \(H\), `element i ∈ set j` ilişkisini doğrudan tutar. `HypergraphConv` node→hyperedge→node mesajlaşmasını bu incidence yapısı üzerinden uygular:

\[
X' = D^{-1} H W B^{-1} H^\top X\Theta
\]

Bu nedenle hypergraph burada yalnız terminoloji değildir; problemdeki tek bir karar değişkeninin birçok elemanı aynı anda bağlaması nedeniyle doğal temsil biçimidir.

Üretim kullanımı için güvenli desen:

```text
Hypergraph GNN → candidate screening → coverage repair → MILP/CP solver → doğrulanmış çözüm
```

Distribution shift, exact-label üretim maliyeti ve güçlü greedy/LP-rounding/solver baseline'ları ayrıca test edilmelidir.